### **Code used to process the mouse brain STARR-FISH data**

##### This notebook provides an example of the code used for cell segmentation, spot filtering, and annData object creation. Spots should first be fitted from the data using the appropriate worker script. Cell type annotation using the imaged marker gene transcripts can be performed using Celltype Mapper from the Allen Institute.

#### **Overview:**

1. Cell segmentation
2. Spot filtering
3. Scanpy annData creation

#### **Key outputs:**
- Cell segmentation masks for the mouse brain
- Final, filtered spots for STARR-FISH/MERFISH data
- h5ad file for the cCRE, T7, and marker gene expression across the mouse brain that includes cell type annotation

### **1. Cell segmentation**

#### *1a. Import image-related functions*

In [ ]:
from ioMicro import *

#### *1b. Define fovs for segmentation*

In [ ]:
all_fls = glob.glob(r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\H1_MER_set*\*.zarr')
fovs = [os.path.basename(fl).split('.')[0]for fl in all_fls]

#### *1c. Perform 3D segmentation with Cellpose*

In [ ]:
for fov in tqdm(fovs):
    iset = int(fov.split('_')[-2][-1])
    fl_ref = rf'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\H9_CREMER_set{iset}'+os.sep+fov
    
    seg_fld = r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\Segmentation'
    if not os.path.exists(seg_fld): os.makedirs(seg_fld)
    flfsave = seg_fld+os.sep+fov+f'--H9_CREMER_set{iset}'+'--segm.npz'
    if not os.path.exists(flfsave):
        im_dapi = np.array(read_im(fl_ref)[-1],dtype=np.float32)
        #napari.view_image(im_dapi)
        flat_field_tag = r'\\192.168.0.146\loquat1\Zane\Zane400Enhancers_12_11_2024_NOT7\MERFISH_Analysis'+os.sep ############################
        
        im_med_fl=flat_field_tag+r'med_col_raw3.npz'
        immed = cv2.blur(np.load(im_med_fl)['im'],(20,20))
        immed=immed/np.median(immed)
        im_dapi = im_dapi/immed
        psf_file = r'\\192.168.0.146\loquat1\Zane\Zane400Enhancers_12_11_2024_NOT7\MERFISH_Analysis\dic_psfff_Zane2800.pkl'
        psf = np.load(psf_file,allow_pickle=True)
        im_dapid = full_deconv(im_dapi,s_=280,
            pad=100,
            psf=psf,
            parameters={'method': 'wiener', 'beta': 0.001, 'niter': 50},
            gpu=True,
            force=True,
        )
        vmax=10000
        vmin=100
        img = np.clip((im_dapid[:,::3,::3]-vmin)/(vmax-vmin),0,1)
        from cellpose import models
        model = models.Cellpose(gpu=True, model_type='nuclei')
        masks, flows, styles, diams = model.eval(img, batch_size=8,
            channels=[0, 0],
            channel_axis=None,
            invert=False,
            normalize=True,
            diameter=25.0,
            do_3D=True,cellprob_threshold=0.,
            flow_threshold=0.25,min_size = 500)
        np.savez_compressed(flfsave,segm=masks,shape=im_dapi.shape);

### **2. Spot filtering**

#### *2a. Filter marker gene spots*

In [ ]:
# function to filter marker gene spots
def filter_spots(fov,redo=False):
    set_ = '_set'+fov.split('_')[-2].split('scan')[-1]
    final_save_folderF = r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\final_spots_combined'
    save_folder = r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\MERFISH_analysis'
    fld_ref_drift = rf'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\H5_MER{set_}'
    fld_segm_raw = rf'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\H9_CREMER{set_}'
    save_folder_ref_segm = r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\CREMERFISH_analysisNoDec'
    segm_fld = r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\Segmentation'
    if not os.path.exists(final_save_folderF): os.makedirs(final_save_folderF)
    final_fl_save = final_save_folderF+os.sep+fov+'--MERcomb.npz'
    if not os.path.exists(final_fl_save) or redo:
        
        dec = decoder_simple(save_folder,fov,set_=set_)
        dec.ncols = 3
        dec.load_decoded()
        dec.dist_best=np.load(dec.decoded_fl)['dist_best']
        # keep = np.sum(dec.XH_pruned[:,:,-3]>5000,axis=-1)>=3
        # #keep&=dec.dist_best>0.25
        # dec.XH_pruned=dec.XH_pruned[keep]
        # dec.dist_best=dec.dist_best[keep]
        # dec.icodesN = dec.icodesN[keep]
        
        score = get_score(dec)
        scores_ref_fl = save_folder+os.sep+'scores_ref.npy'
        if not os.path.exists(scores_ref_fl):
            
            score_ref = np.sort(dec.score,axis=0)
            dec.score_ref = score_ref
            np.save(scores_ref_fl,score_ref)
        else:
            dec.score_ref = np.load(scores_ref_fl)
        set_scoreA(dec)
        
        dec.th = -0.75
        keep = dec.scoreA>dec.th
        icodesN = dec.icodesN[keep]
        XH_pruned = dec.XH_pruned[keep]
        Xpr = np.nanmean(XH_pruned,axis=1)
        Xmol = Xpr[:,:3]
        
        #correct drift
        fl = fld_ref_drift+os.sep+fov
        save_folder = dec.save_folder
        fl_ref = fld_segm_raw+os.sep+fov
        drift = get_best_translation_pointsV2(fl,fl_ref,save_folder,save_folder_ref_segm,set_=set_,resc=5,th=3)
        
        Xmol=Xmol+drift[0]
        
        ### load segmentation
        seg_fl = segm_fld+fr'\{fov}--H9_CREMER{set_}--segm.npz'
        segm = np.load(seg_fl)['segm']
        shape = np.load(seg_fl)['shape']
        
        XR = Xmol*segm.shape/shape
        dec.XR = XR
        dec.segm = segm
        
        ### asign to cells
        Xcells = np.array(np.where(segm>0)).T
        from scipy.spatial import KDTree
        tree = KDTree(Xcells)
        dd,ii  = tree.query(XR)
        icellsS = segm[tuple(Xcells.T)]
        keep = dd<10
        index_cells = icellsS[ii[keep]].astype(np.int64)
        
        genes= dec.gns_names
        igenes= np.arange(len(genes))
        cellsu = np.unique(icellsS)
        M = np.zeros([len(cellsu),len(genes)],dtype=np.float32)
        Mgn = np.max(igenes)+1
        iVALS,iCTS = np.unique((index_cells-1)*Mgn+icodesN[keep],return_counts=True)
        M[iVALS//Mgn,iVALS%Mgn]=iCTS
        
        np.savez_compressed(final_fl_save,Mmer=M,genes_mer = genes,cells_mer = [fov+'--'+str(e) for e in cellsu])
        return dec

In [ ]:
dec_fls = glob.glob(r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\MERFISH_analysis\decoded*')

fovs = np.random.permutation([os.path.basename(fl).replace('.zarr','') 
        for fl in glob.glob(r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\H9_CREMER*\*.zarr')])

fovs_comp = [fl.split('decodedNew_')[-1].split('--')[0]for fl in dec_fls]

In [ ]:
for fov in tqdm(fovs):
    try:
        filter_spots(fov,redo=False)
    except:
        print("Failed:",fov)

#### *2b. Filter cCRE spots*

##### *2b.1. Decode SYFP2 spots to use for cCRE filtering*

In [ ]:
import glob

save_folder = r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\CREMERFISH_analysisNoDec'
dec_fls = glob.glob(save_folder+os.sep+'decoded*')
all_zarrs = glob.glob(r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\H9_CREMER*\*.zarr')

def get_fov(fl):
    return os.path.basename(fl).replace('decodedNew_','').replace('.zarr','').split('--')[0]

fovs = [get_fov(fl) for fl in all_zarrs]

GFP_fls = glob.glob(r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\CREMERFISH_analysisNoDec\*--*GFP*--col1__Xhfits.npz')

In [ ]:
from scipy.spatial import KDTree

save_folder_CRE = r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\CREMERFISH_analysisNoDec'

def im_max__from_Xh(Xh,ds=5,M=None):
    X = Xh[:,1:3]
    H = Xh[:,-1]
    
    if M is None:
        m = np.min(X,axis=0)
        X = X-m
        M = np.max(X,axis=0).astype(int)+2
    else:
        M=np.array(M)
        keep =np.all((X<M)&(X>=0),axis=-1) 
        X = X[keep]
        H=H[keep]
    X = X.astype(int)
    im_max = np.zeros(M,dtype=np.float32)
    im_max[tuple(X.T)]=H
    im_norm= np.zeros([10*ds]*2,dtype=np.float32)
    im_norm[ds*5,ds*5]=1
    hrescale = np.max(cv2.GaussianBlur(im_norm, (ds,ds),0))
    im_max_ = cv2.GaussianBlur(im_max, (ds,ds),0)/hrescale
    return im_max_

def get_match(Xh0T,Xh1T,cutoff=2,plt_val=False):
    #X,Y = Xh0T[:,:3],Xh1T[:,:3]
    HX,HY = Xh0T[:,-1],Xh1T[:,-1]
    iX = np.argsort(HX)#[::-1]
    iY = np.argsort(HY)#[::-1]
    Xh0,Xh1 = Xh0T[iX],Xh1T[iY]
    X,Y = Xh0[:,:3],Xh1[:,:3]
    HX,HY = Xh0[:,-1],Xh1[:,-1]
    
    # Build kd-trees for fast spatial queries:
    treeX = KDTree(X)
    
    treeY = KDTree(Y)
    
    candidate_indices = treeX.query_ball_tree(treeY, cutoff)
    dic1 = {iY:iX for iY,iXs in enumerate(candidate_indices) for iX in iXs}
    dic2 = {dic1[iY]:iY for iY in dic1}
    iXh1 = list(dic2.keys())
    iXh0 = [dic2[key] for key in iXh1]
    Xh0_,Xh1_ = Xh0[iXh0],Xh1[iXh1]
    unpaired0 = np.setdiff1d(np.arange(len(Xh0)),iXh0)
    if plt_val:
        import napari
        V= napari.view_image(im_max__from_Xh(Xh0_[:,:8]))
        V.add_image(im_max__from_Xh(Xh1_[:,:8]))
    return np.array([Xh0_,Xh1_]).swapaxes(0,1)

def get_gfp_spots_dec(fov,plt_val=False,redo=False):
    set_ = '_set'+fov.split('_')[-2].split('scan')[-1]
    save_folder = os.path.dirname(save_folder_CRE)+os.sep+'final_spots'#r'L:\Zane\Zane400Enhancers_12_11_2024_NOT7\final_spots'
    if not os.path.exists(save_folder): os.makedirs(save_folder)
    saveflf = save_folder+os.sep+fov+'--decspots.npz'
    if not os.path.exists(saveflf) or redo:
        gfp_fl0 = save_folder_CRE+os.sep+rf'\{fov}--HGFP{set_}--col0__Xhfits.npz'
        gfp_fl1 = save_folder_CRE+os.sep+rf'\{fov}--HGFP{set_}--col1__Xhfits.npz'
        gfp_fl2 = save_folder_CRE+os.sep+rf'\{fov}--HGFP{set_}--col2__Xhfits.npz'
        #\Conv_zscan1_120--HGFP_set1--col1__Xhfits.npz
        Xh0 = np.load(gfp_fl0)['Xh']
        Xh1 = np.load(gfp_fl1)['Xh']
        Xh0[:,:3] = Xh0[:,:3]-[ 0.8456841 , -0.24446984, -0.19531506] ### bring to Cy5 channel
        keep = (Xh0[:,-2]>0.5)#(Xh0[:,-3]>np.exp(10))|
        Xh0 = Xh0[keep]
        keep = (Xh1[:,-2]>0.5)#(Xh1[:,-3]>np.exp(10))|
        Xh1 = Xh1[keep]


        X_sat0 = np.load(gfp_fl0)['X_sat']
        X_sat1 = np.load(gfp_fl1)['X_sat']
        
        Xh2 = np.load(gfp_fl2)['Xh']
        X_sat = np.concatenate([X_sat0,X_sat1],axis=0)
        tree = KDTree(X_sat)
        dd,nn = tree.query(Xh2[:,:3])
        Xh2_sat = Xh2[dd<3]
        Xh2_sat = Xh2_sat[Xh2_sat[:,-2]>0.5]
        XGFP = get_match(Xh0,Xh1,cutoff=2)
        
        if False: ### check the 2 dapi images
            im0  = im_max__from_Xh(Xh0,ds=3,M=[2800,2800])
            im1  = im_max__from_Xh(Xh1,ds=3,M=[2800,2800])
            import napari
            V = napari.view_image(im0)
            V.add_image(im1)
    
        drift_fl = save_folder_CRE+os.sep+fr'\driftNew_{fov}--{set_}.pkl'
        #driftNew_Conv_zscan1_120--_set1.pkl
        drifts,flds,fov_,fl_ref = np.load(drift_fl,allow_pickle=True)
        dic_drift = {os.path.basename(fld):drift[0] for fld,drift in zip(flds,drifts)}
        #get_match(np.random.rand(100,4),np.random.rand(100,4),cutoff=20).shape
        print(dic_drift)
        #print(XGFP)
        gfptag = [key for key in dic_drift if 'GFP' in key.upper()][0]
        XGFP[...,:3]=XGFP[...,:3]+dic_drift[gfptag][np.newaxis,np.newaxis]
        Xh2_sat[...,:3]=Xh2_sat[...,:3]+dic_drift[gfptag][np.newaxis]
        XGFP_ = np.mean(XGFP,axis=1)
        XGFP_ = np.concatenate([XGFP_,Xh2_sat])
        
        dec_fl = save_folder_CRE+os.sep+rf'\decodedNew_{fov}--{set_}.npz'
        list(np.load(dec_fl).keys())
        XH_pruned= np.load(dec_fl)['XH_pruned']
        icodesN= np.load(dec_fl)['icodesN']
        gns_names= np.load(dec_fl)['gns_names']
        dist_best= np.load(dec_fl)['dist_best']
        Xpr = np.nanmean(XH_pruned,axis=1)
        
        dd,ii = KDTree(Xpr[:,:3]).query(XGFP_[:,:3],5)
        ii[ii>=len(Xpr)]=-1
        Hpr = Xpr[:,-3][ii]
        bad = dd>2
        Hpr[bad]=0
        bad = np.all(bad,axis=-1)
        iibest = ii[np.arange(len(ii)),np.argmax(Hpr,axis=-1)]
        iibest = iibest[~bad]
        ikeep = iibest>-1
        #ikeep = (dist_best[iibest]<0.3)&(Xpr[:,-3][iibest]>5000)
        
        iibest = iibest[ikeep]
        XGFPkeep = XGFP_[~bad][ikeep]
        gns_F = gns_names[icodesN[iibest]]
        XH_pruned_F = XH_pruned[iibest]
        dist_best_F = dist_best[iibest]
    
        np.savez(saveflf,XGFPkeep=XGFPkeep,XGFP_=XGFP_,XGFP=XGFP,gns_F=gns_F,XH_pruned_F=XH_pruned_F,dist_best_F=dist_best_F)

    print(saveflf)
   
    if plt_val:
        XH_pruned_F = np.load(saveflf)['XH_pruned_F']
        dist_best_F = np.load(saveflf)['dist_best_F']
        XGFPkeep = np.load(saveflf)['XGFPkeep']
        gns_F = np.load(saveflf)['gns_F']
        XGFP = np.load(saveflf)['XGFP']
        
        Xpr = np.nanmean(XH_pruned_F,axis=1)
        ikeep = (dist_best_F<0.5)&(Xpr[:,-3]>5000)
        XGFPkeep_ = XGFPkeep[ikeep]
        gns_F_ = gns_F[ikeep]
        
        imGFP  = im_max__from_Xh(XGFP_,ds=3,M=[2800,2800])
        #imGFPF  = im_max__from_Xh(XGFPkeep_,ds=3,M=[2800,2800])
        import napari
        V = napari.Viewer()
        V.add_image(imGFP,contrast_limits=[0,30000])
        #V.add_points(np.mean(XGFP,axis=1)[:,1:3],size=7)
        V.add_points(XGFPkeep[:,1:3],face_color=[0]*4,edge_color='r')
        V.add_points(XGFPkeep_[:,1:3],face_color=[0]*4,edge_color='g')
        # for gn in np.unique(gns_F_):
        #    V.add_points(XGFPkeep_[gns_F_==gn][:,1:3],face_color=[0]*4,edge_color=np.random.rand(3),name=gn)
        return V

In [ ]:
for fov in tqdm(fovs):
    try:
        get_gfp_spots_dec(fov,plt_val=False)
    except:
        print("Failed",fov)

##### *2b.2. Filter cCRE spots using decoded SYFP2 spots*

In [ ]:
# function to filter cCRE spots
def get_cre_spots(fov):
    final_fl_save = rf'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\final_spots_combined\{fov}--CREcomb2.npz'
    if not os.path.exists(final_fl_save):
        fl_dec = fr'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\final_spots\{fov}--decspots.npz'
        dic_dec = np.load(fl_dec)
        gns_F = dic_dec['gns_F']
        XH_pruned_F = dic_dec['XH_pruned_F']
        XGFP_F = dic_dec['XGFPkeep']
        dist_best_F = dic_dec['dist_best_F']
        H = np.nanmean(XH_pruned_F[...,-3],axis=1)
        keep = (H>2500)&(dist_best_F<0.5)
        gns_F = gns_F[keep]
        XGFP_F = XGFP_F[keep]
        
        Xmol = XGFP_F[:,:3]
        
        ### load segmentation
        #Conv_zscan1_004--H9_CREMER_set1--segm.npz
        iset = fov.split('_')[-2].split('scan')[-1]
        seg_fl = rf'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\Segmentation\{fov}--H9_CREMER_set{iset}--segm.npz'
        segm = np.load(seg_fl)['segm']
        shape = np.load(seg_fl)['shape']
        
        XR = Xmol*segm.shape/shape
        
        #gns_names = np.load(r'L:\Zane\Zane400Enhancers_12_11_2024_NOT7\CREMERFISH_analysisNoDec\decodedNew_Conv_zscan1_047--.npz')['gns_names']
        #print(list(gns_names))
        gns_names = np.array(['CRE001', 'CRE002', 'CRE003', 'CRE004', 'CRE005', 'CRE006', 'CRE007', 'CRE008', 'CRE009', 'CRE010', 'CRE011', 'CRE012', 'CRE013', 'CRE014', 'CRE015', 'CRE016', 'CRE017', 'CRE018', 'CRE019', 'CRE020', 'CRE021', 'CRE022', 'CRE023', 'CRE024', 'CRE025', 'CRE026', 'CRE027', 'CRE028', 'CRE029', 'CRE030', 'CRE031', 'CRE032', 'CRE033', 'CRE034', 'CRE035', 'CRE036', 'CRE037', 'CRE038', 'CRE039', 'CRE040', 'CRE041', 'CRE042', 'CRE043', 'CRE044', 'CRE045', 'CRE046', 'CRE047', 'CRE048', 'CRE049', 'CRE050', 'CRE051', 'CRE052', 'CRE053', 'CRE054', 'CRE055', 'CRE056', 'CRE057', 'CRE058', 'CRE059', 'CRE060', 'CRE061', 'CRE062', 'CRE063', 'CRE064', 'CRE065', 'CRE066', 'CRE067', 'CRE068', 'CRE069', 'CRE070', 'CRE071', 'CRE072', 'CRE073', 'CRE074', 'CRE075', 'CRE076', 'CRE077', 'CRE078', 'CRE079', 'CRE080', 'CRE081', 'CRE082', 'CRE083', 'CRE084', 'CRE085', 'CRE086', 'CRE087', 'CRE088', 'CRE089', 'CRE090', 'CRE091', 'CRE092', 'CRE093', 'CRE094', 'CRE095', 'CRE096', 'CRE097', 'CRE098', 'CRE099', 'CRE100', 'CRE101', 'CRE102', 'CRE103', 'CRE104', 'CRE105', 'CRE106', 'CRE107', 'CRE108', 'CRE109', 'CRE110', 'CRE111', 'CRE112', 'CRE113', 'CRE114', 'CRE115', 'CRE116', 'CRE117', 'CRE118', 'CRE119', 'CRE120', 'CRE121', 'CRE122', 'CRE123', 'CRE124', 'CRE125', 'CRE126', 'CRE127', 'CRE128', 'CRE129', 'CRE130', 'CRE131', 'CRE132', 'CRE133', 'CRE134', 'CRE135', 'CRE136', 'CRE137', 'CRE138', 'CRE139', 'CRE140', 'CRE141', 'CRE142', 'CRE143', 'CRE144', 'CRE145', 'CRE146', 'CRE147', 'CRE148', 'CRE149', 'CRE150', 'CRE151', 'CRE152', 'CRE153', 'CRE154', 'CRE155', 'CRE156', 'CRE157', 'CRE158', 'CRE159', 'CRE160', 'CRE161', 'CRE162', 'CRE163', 'CRE164', 'CRE165', 'CRE166', 'CRE167', 'CRE168', 'CRE169', 'CRE170', 'CRE171', 'CRE172', 'CRE173', 'CRE174', 'CRE175', 'CRE176', 'CRE177', 'CRE178', 'CRE179', 'CRE180', 'CRE181', 'CRE182', 'CRE183', 'CRE184', 'CRE185', 'CRE186', 'CRE187', 'CRE188', 'CRE189', 'CRE190', 'CRE191', 'CRE192', 'CRE193', 'CRE194', 'CRE195', 'CRE196', 'CRE197', 'CRE198', 'CRE199', 'CRE200', 'CRE201', 'CRE202', 'CRE203', 'CRE204', 'CRE205', 'CRE206', 'CRE207', 'CRE208', 'CRE209', 'CRE210', 'CRE211', 'CRE212', 'CRE213', 'CRE214', 'CRE215', 'CRE216', 'CRE217', 'CRE218', 'CRE219', 'CRE220', 'CRE221', 'CRE222', 'CRE223', 'CRE224', 'CRE225', 'CRE226', 'CRE227', 'CRE228', 'CRE229', 'CRE230', 'CRE231', 'CRE232', 'CRE233', 'CRE234', 'CRE235', 'CRE236', 'CRE237', 'CRE238', 'CRE239', 'CRE240', 'CRE241', 'CRE242', 'CRE243', 'CRE244', 'CRE245', 'CRE246', 'CRE247', 'CRE248', 'CRE249', 'CRE250', 'CRE251', 'CRE252', 'CRE253', 'CRE254', 'CRE255', 'CRE256', 'CRE257', 'CRE258', 'CRE259', 'CRE260', 'CRE261', 'CRE262', 'CRE263', 'CRE264', 'CRE265', 'CRE266', 'CRE267', 'CRE268', 'CRE269', 'CRE270', 'CRE271', 'CRE272', 'CRE273', 'CRE274', 'CRE275', 'CRE276', 'CRE277', 'CRE278', 'CRE279', 'CRE280', 'CRE281', 'CRE282', 'CRE283', 'CRE284', 'CRE285', 'CRE286', 'CRE287', 'CRE288', 'CRE289', 'CRE290', 'CRE291', 'CRE292', 'CRE293', 'CRE294', 'CRE295', 'CRE296', 'CRE297', 'CRE298', 'CRE299', 'CRE300', 'CRE301', 'CRE302', 'CRE303', 'CRE304', 'CRE305', 'CRE306', 'CRE307', 'CRE308', 'CRE309', 'CRE310', 'CRE311', 'CRE312', 'CRE313', 'CRE314', 'CRE315', 'CRE316', 'CRE317', 'CRE318', 'CRE319', 'CRE320', 'CRE321', 'CRE322', 'CRE323', 'CRE324', 'CRE325', 'CRE326', 'CRE327', 'CRE328', 'CRE329', 'CRE330', 'CRE331', 'CRE332', 'CRE333', 'CRE334', 'CRE335', 'CRE336', 'CRE337', 'CRE338', 'CRE339', 'CRE340', 'CRE341', 'CRE342', 'CRE343', 'CRE344', 'CRE345', 'CRE346', 'CRE347', 'CRE348', 'CRE349', 'CRE350', 'CRE351', 'CRE352', 'CRE353', 'CRE354', 'CRE355', 'CRE356', 'CRE357', 'CRE358', 'CRE359', 'CRE360', 'CRE361', 'CRE362', 'CRE363', 'CRE364', 'CRE365', 'CRE366', 'CRE367', 'CRE368', 'CRE369', 'CRE370', 'CRE371', 'CRE372', 'CRE373', 'CRE374', 'CRE375', 'CRE376', 'CRE377', 'CRE378', 'CRE379', 'CRE380', 'CRE381', 'CRE382', 'CRE383', 'CRE384', 'CRE385', 'CRE386', 'CRE387', 'CRE388', 'CRE389', 'CRE390', 'CRE391', 'CRE392', 'CRE393', 'CRE394', 'CRE395', 'CRE396', 'CRE397', 'CRE398', 'CRE399', 'CRE400', 'CRE401', 'CRE402', 'blank001', 'blank002', 'blank003', 'blank004', 'blank005', 'blank006', 'blank007', 'blank008', 'blank009', 'blank010', 'blank011', 'blank012', 'blank013', 'blank014', 'blank015', 'blank016', 'blank017', 'blank018'])
        gns_names_, inv = np.unique(list(gns_F)+list(gns_names),return_inverse=True)
        icodesN = inv[:len(gns_F)]
        #icodes[keep]
        
        ### asign to cells
        Xcells = np.array(np.where(segm>0)).T
        from scipy.spatial import KDTree
        tree = KDTree(Xcells)
        dd,ii  = tree.query(XR)
        icellsS = segm[tuple(Xcells.T)]
        keep = dd<10
        index_cells = icellsS[ii[keep]].astype(np.int64)
        
        genes= gns_names
        igenes= np.arange(len(genes))
        cellsu = np.unique(icellsS)
        M = np.zeros([len(cellsu),len(genes)],dtype=np.float32)
        Mgn = np.max(igenes)+1
        iVALS,iCTS = np.unique((index_cells-1)*Mgn+icodesN[keep],return_counts=True)
        M[iVALS//Mgn,iVALS%Mgn]=iCTS
        
        np.savez_compressed(final_fl_save,Mmer=M,genes_mer = genes,cells_mer = [fov+'--'+str(e) for e in cellsu])


In [ ]:
dec_fls = glob.glob(r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\final_spots\*--decspots.npz')
fovs = [os.path.basename(fl).split('--')[0] for fl in dec_fls]

for fov in tqdm(fovs):
    try:
        get_cre_spots(fov)
    except:
        print("Failed",fov)

#### *2c. Filter T7 spots*

##### *2c.1. Pre-filter T7 spots*

In [ ]:
import glob

save_folder = r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\TMERFISH_analysisNoDec'
dec_fls = glob.glob(save_folder+os.sep+'decoded*')
all_zarrs = glob.glob(r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\H9_TMER*\*.zarr')

def get_fov(fl):
    return os.path.basename(fl).replace('decodedNew_','').replace('.zarr','').split('--')[0]

fovs = [get_fov(fl) for fl in all_zarrs]

In [ ]:
from scipy.spatial import KDTree

save_folder_CRE = r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\TMERFISH_analysisNoDec'

def filter_TCRE(fov,redo=False):
    set_ = '_set'+fov.split('_')[-2].split('scan')[-1]
    
    save_folder = os.path.dirname(save_folder_CRE)+os.sep+'final_spots_TMER'
    if not os.path.exists(save_folder): os.makedirs(save_folder)
    saveflf = save_folder+os.sep+fov+'--decspots.npz'
    if not os.path.exists(saveflf) or redo:
        dec_fl = save_folder_CRE+os.sep+rf'\decodedNew_{fov}--{set_}.npz'
        list(np.load(dec_fl).keys())
        XH_pruned= np.load(dec_fl)['XH_pruned']
        icodesN= np.load(dec_fl)['icodesN']
        gns_names= np.load(dec_fl)['gns_names']
        dist_best= np.load(dec_fl)['dist_best']
        Xpr = np.nanmean(XH_pruned,axis=1)
        
        icols = np.nanmean(XH_pruned[...,-2],axis=1).astype(int)
        ucols = np.unique(icols)
        
        H = np.nanmedian(XH_pruned[...,-3],axis=1)
        #D = dec.XH_pruned[...,:3]-np.nanmean(dec.XH_pruned[...,:3],axis=1)[:,np.newaxis]
        #D = np.nanmean(np.linalg.norm(D,axis=-1),axis=-1)
        n1bits = XH_pruned.shape[1]
        from itertools import combinations
        combs = np.array(list(combinations(np.arange(n1bits),2)))
        X = XH_pruned[:,:,:3]
        D = np.nanmean(np.linalg.norm(X[:,combs][:,:,0]-X[:,combs][:,:,1],axis=-1),axis=1)
        
        db = dist_best
        score = np.array([H,-D,-db]).T
        
        
        scoresRef_fl = save_folder_CRE+os.sep+'scoresRef.pkl'
        if not os.path.exists(scoresRef_fl):
            scoresRef = [np.sort(score[icols==icol],axis=0) for icol in ucols]
            pickle.dump(scoresRef,open(scoresRef_fl,'wb'))
        else:
            scoresRef = pickle.load(open(scoresRef_fl,'rb'))
        
        keep_color = [icols==icol for icol in ucols]
        scoreA = np.zeros(len(H))
        for icol in ucols:
            scoresRef_ = scoresRef[icol]
            score_ = score[keep_color[icol]]
            from scipy.spatial import KDTree
            scoreA_ = np.zeros(len(score_))
            iSs = np.arange(scoresRef_.shape[-1])
            for iS in iSs:
                dist_,inds_ = KDTree(scoresRef_[:,[iS]]).query(score_[:,[iS]])
                scoreA_+=np.log((inds_+1))-np.log(len(scoresRef_))
            scoreA[keep_color[icol]]=scoreA_
        
        keep = scoreA>-1
        gns_F = gns_names[icodesN[keep]]
        XH_pruned_F = XH_pruned[keep]
        dist_best_F = dist_best[keep]
        
        np.savez(saveflf,gns_F=gns_F,XH_pruned_F=XH_pruned_F,dist_best_F=dist_best_F)

In [ ]:
missing_fovs = []
for fov in tqdm(fovs):
    try:
        filter_TCRE(fov,redo=False)
    except:
        missing_fovs.append(fov)

##### *2c.2. Filter T7 spots using pre-filtered spots*

In [ ]:
# function to get filtered T7 spots
def get_tcre_spots(fov,redo=False):
    final_fl_save = rf'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\final_spots_combined\{fov}--TCREcomb3.npz'
    if not os.path.exists(final_fl_save) or redo:
        fl_dec = fr'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\final_spots_TMER\{fov}--decspots.npz'
        dic_dec = np.load(fl_dec)
        gns_F = dic_dec['gns_F']
        XH_pruned_F = dic_dec['XH_pruned_F']
        
        dist_best_F = dic_dec['dist_best_F']
        H = np.nanmean(XH_pruned_F[...,-3],axis=1)
        keep = (H>2500)&(dist_best_F<0.5)
        
        XGFP_F = np.nanmean(XH_pruned_F,axis=1)
        gns_F = gns_F[keep]
        XGFP_F = XGFP_F[keep]
        
        Xmol = XGFP_F[:,:3]
        
        ### load segmentation
        #Conv_zscan1_004--H9_CREMER_set1--segm.npz
        iset = fov.split('_')[-2].split('scan')[-1]
        fl_ref = rf'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\TMERFISH_analysisNoDec\{fov}--H9_TMER_set{iset}--_set{iset}dapiFeatures.npz'
        fl_segf =rf'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\CREMERFISH_analysisNoDec\{fov}--H9_CREMER_set{iset}--_set{iset}dapiFeatures.npz'
        Xh_ref = np.load(fl_ref)['Xh_plus']
        Xh_segf = np.load(fl_segf)['Xh_plus']
        tzxy = get_best_translation_points( Xh_ref[:,:3],
            Xh_segf[:,:3],
            resc=5,
            target=3,
            return_counts=False,
        )
        
        iset = fov.split('_')[-2].split('scan')[-1]
        seg_fl = rf'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\Segmentation\{fov}--H9_CREMER_set{iset}--segm.npz'
        segm = np.load(seg_fl)['segm']
        shape = np.load(seg_fl)['shape']
        
        XR = (Xmol-tzxy)*segm.shape/shape
        
        #gns_names = np.load(r'L:\Zane\Zane400Enhancers_12_11_2024_NOT7\CREMERFISH_analysisNoDec\decodedNew_Conv_zscan1_047--.npz')['gns_names']
        #print(list(gns_names))
        gns_names = np.array(['CRE001', 'CRE002', 'CRE003', 'CRE004', 'CRE005', 'CRE006', 'CRE007', 'CRE008', 'CRE009', 'CRE010', 'CRE011', 'CRE012', 'CRE013', 'CRE014', 'CRE015', 'CRE016', 'CRE017', 'CRE018', 'CRE019', 'CRE020', 'CRE021', 'CRE022', 'CRE023', 'CRE024', 'CRE025', 'CRE026', 'CRE027', 'CRE028', 'CRE029', 'CRE030', 'CRE031', 'CRE032', 'CRE033', 'CRE034', 'CRE035', 'CRE036', 'CRE037', 'CRE038', 'CRE039', 'CRE040', 'CRE041', 'CRE042', 'CRE043', 'CRE044', 'CRE045', 'CRE046', 'CRE047', 'CRE048', 'CRE049', 'CRE050', 'CRE051', 'CRE052', 'CRE053', 'CRE054', 'CRE055', 'CRE056', 'CRE057', 'CRE058', 'CRE059', 'CRE060', 'CRE061', 'CRE062', 'CRE063', 'CRE064', 'CRE065', 'CRE066', 'CRE067', 'CRE068', 'CRE069', 'CRE070', 'CRE071', 'CRE072', 'CRE073', 'CRE074', 'CRE075', 'CRE076', 'CRE077', 'CRE078', 'CRE079', 'CRE080', 'CRE081', 'CRE082', 'CRE083', 'CRE084', 'CRE085', 'CRE086', 'CRE087', 'CRE088', 'CRE089', 'CRE090', 'CRE091', 'CRE092', 'CRE093', 'CRE094', 'CRE095', 'CRE096', 'CRE097', 'CRE098', 'CRE099', 'CRE100', 'CRE101', 'CRE102', 'CRE103', 'CRE104', 'CRE105', 'CRE106', 'CRE107', 'CRE108', 'CRE109', 'CRE110', 'CRE111', 'CRE112', 'CRE113', 'CRE114', 'CRE115', 'CRE116', 'CRE117', 'CRE118', 'CRE119', 'CRE120', 'CRE121', 'CRE122', 'CRE123', 'CRE124', 'CRE125', 'CRE126', 'CRE127', 'CRE128', 'CRE129', 'CRE130', 'CRE131', 'CRE132', 'CRE133', 'CRE134', 'CRE135', 'CRE136', 'CRE137', 'CRE138', 'CRE139', 'CRE140', 'CRE141', 'CRE142', 'CRE143', 'CRE144', 'CRE145', 'CRE146', 'CRE147', 'CRE148', 'CRE149', 'CRE150', 'CRE151', 'CRE152', 'CRE153', 'CRE154', 'CRE155', 'CRE156', 'CRE157', 'CRE158', 'CRE159', 'CRE160', 'CRE161', 'CRE162', 'CRE163', 'CRE164', 'CRE165', 'CRE166', 'CRE167', 'CRE168', 'CRE169', 'CRE170', 'CRE171', 'CRE172', 'CRE173', 'CRE174', 'CRE175', 'CRE176', 'CRE177', 'CRE178', 'CRE179', 'CRE180', 'CRE181', 'CRE182', 'CRE183', 'CRE184', 'CRE185', 'CRE186', 'CRE187', 'CRE188', 'CRE189', 'CRE190', 'CRE191', 'CRE192', 'CRE193', 'CRE194', 'CRE195', 'CRE196', 'CRE197', 'CRE198', 'CRE199', 'CRE200', 'CRE201', 'CRE202', 'CRE203', 'CRE204', 'CRE205', 'CRE206', 'CRE207', 'CRE208', 'CRE209', 'CRE210', 'CRE211', 'CRE212', 'CRE213', 'CRE214', 'CRE215', 'CRE216', 'CRE217', 'CRE218', 'CRE219', 'CRE220', 'CRE221', 'CRE222', 'CRE223', 'CRE224', 'CRE225', 'CRE226', 'CRE227', 'CRE228', 'CRE229', 'CRE230', 'CRE231', 'CRE232', 'CRE233', 'CRE234', 'CRE235', 'CRE236', 'CRE237', 'CRE238', 'CRE239', 'CRE240', 'CRE241', 'CRE242', 'CRE243', 'CRE244', 'CRE245', 'CRE246', 'CRE247', 'CRE248', 'CRE249', 'CRE250', 'CRE251', 'CRE252', 'CRE253', 'CRE254', 'CRE255', 'CRE256', 'CRE257', 'CRE258', 'CRE259', 'CRE260', 'CRE261', 'CRE262', 'CRE263', 'CRE264', 'CRE265', 'CRE266', 'CRE267', 'CRE268', 'CRE269', 'CRE270', 'CRE271', 'CRE272', 'CRE273', 'CRE274', 'CRE275', 'CRE276', 'CRE277', 'CRE278', 'CRE279', 'CRE280', 'CRE281', 'CRE282', 'CRE283', 'CRE284', 'CRE285', 'CRE286', 'CRE287', 'CRE288', 'CRE289', 'CRE290', 'CRE291', 'CRE292', 'CRE293', 'CRE294', 'CRE295', 'CRE296', 'CRE297', 'CRE298', 'CRE299', 'CRE300', 'CRE301', 'CRE302', 'CRE303', 'CRE304', 'CRE305', 'CRE306', 'CRE307', 'CRE308', 'CRE309', 'CRE310', 'CRE311', 'CRE312', 'CRE313', 'CRE314', 'CRE315', 'CRE316', 'CRE317', 'CRE318', 'CRE319', 'CRE320', 'CRE321', 'CRE322', 'CRE323', 'CRE324', 'CRE325', 'CRE326', 'CRE327', 'CRE328', 'CRE329', 'CRE330', 'CRE331', 'CRE332', 'CRE333', 'CRE334', 'CRE335', 'CRE336', 'CRE337', 'CRE338', 'CRE339', 'CRE340', 'CRE341', 'CRE342', 'CRE343', 'CRE344', 'CRE345', 'CRE346', 'CRE347', 'CRE348', 'CRE349', 'CRE350', 'CRE351', 'CRE352', 'CRE353', 'CRE354', 'CRE355', 'CRE356', 'CRE357', 'CRE358', 'CRE359', 'CRE360', 'CRE361', 'CRE362', 'CRE363', 'CRE364', 'CRE365', 'CRE366', 'CRE367', 'CRE368', 'CRE369', 'CRE370', 'CRE371', 'CRE372', 'CRE373', 'CRE374', 'CRE375', 'CRE376', 'CRE377', 'CRE378', 'CRE379', 'CRE380', 'CRE381', 'CRE382', 'CRE383', 'CRE384', 'CRE385', 'CRE386', 'CRE387', 'CRE388', 'CRE389', 'CRE390', 'CRE391', 'CRE392', 'CRE393', 'CRE394', 'CRE395', 'CRE396', 'CRE397', 'CRE398', 'CRE399', 'CRE400', 'CRE401', 'CRE402', 'blank001', 'blank002', 'blank003', 'blank004', 'blank005', 'blank006', 'blank007', 'blank008', 'blank009', 'blank010', 'blank011', 'blank012', 'blank013', 'blank014', 'blank015', 'blank016', 'blank017', 'blank018'])
        gns_names_, inv = np.unique(list(gns_F)+list(gns_names),return_inverse=True)
        icodesN = inv[:len(gns_F)]
        #icodes[keep]
        
        ### asign to cells
        Xcells = np.array(np.where(segm>0)).T
        from scipy.spatial import KDTree
        tree = KDTree(Xcells)
        dd,ii  = tree.query(XR)
        icellsS = segm[tuple(Xcells.T)]
        keep = dd<10
        index_cells = icellsS[ii[keep]].astype(np.int64)
        
        
        genes= gns_names
        igenes= np.arange(len(genes))
        cellsu = np.unique(icellsS)
        M = np.zeros([len(cellsu),len(genes)],dtype=np.float32)
        Mgn = np.max(igenes)+1
        iVALS,iCTS = np.unique((index_cells-1)*Mgn+icodesN[keep],return_counts=True)
        M[iVALS//Mgn,iVALS%Mgn]=iCTS
        
        np.savez_compressed(final_fl_save,Mmer=M,genes_mer = genes,cells_mer = [fov+'--'+str(e) for e in cellsu])
        return XR,segm

In [ ]:
dec_fls = glob.glob(r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\final_spots_TMER\*--decspots.npz')
fovs = [os.path.basename(fl).split('--')[0] for fl in dec_fls]

missing = []
for fov in tqdm(fovs):
    try:
        get_tcre_spots(fov,redo=False)
    except:
        missing.append(fov)

### **3. Scanpy annData creation**

#### *3a. Collect and filter cell info from image data*

In [ ]:
def get_fov(fl):
    return os.path.basename(fl).split('--')[0].split('.')[0]
def get_iset(fov):
    return int(fov.split('_')[-2].split('scan')[-1])

segm_fld = r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\Segmentation'
fls = np.sort(glob.glob(segm_fld+r'\*-segm.npz'))
raw_fls = np.sort( glob.glob(r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\H9_CREMER*\*.zarr'))

dic_fls = {get_fov(fl):fl for fl in fls}
dic_raw = {get_fov(fl):fl for fl in raw_fls}

In [ ]:
from scipy import ndimage as ndi

cmsf = []
volmsf = []
ifovsf = []
pos_fov = []
cellsf = []
icells = []

for fov in tqdm(list(dic_fls.keys())):
    fl = dic_fls[fov]
    fl_raw = dic_raw[fov]
    iset = get_iset(fov)
    ifov = get_ifov(fov)+10000*iset
    
    if ifov not in ifovsf:
        dic = np.load(fl)
        segm = dic['segm']
        shape = dic['shape']
        
        cells,volms = np.unique(segm,return_counts=True)
        cells,volms = cells[1:],volms[1:]
        cms = np.array(ndi.center_of_mass(segm>0,segm,cells))
        icells.extend(cells)

        if len(cms):
            pix_size = shape/segm.shape*[0.5,0.108333,0.108333]
            cms_um = cms*pix_size
            ifovs = [ifov]*len(cells)
            pos_= [read_im(fl_raw,return_pos=True)[1:]]*len(cells)
            cmsf.extend(cms_um)
            volmsf.extend(volms)
            pos_fov.extend(pos_)
            ifovsf.extend(ifovs)
            
# save the cell info            
np.savez(r"\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\cellinfos.npz",cmsf=cmsf,volmsf=volmsf,pos_fov=pos_fov,ifovsf=ifovsf,icells=icells)

In [ ]:
from scipy.spatial import KDTree

cellsinfo = np.load(r"\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\cellinfos.npz")
cmsf = cellsinfo['cmsf']
volmsf = cellsinfo['volmsf']
pos_fov = cellsinfo['pos_fov']
ifovsf = cellsinfo['ifovsf']
icells = cellsinfo['icells']
transpose,flipx,flipy=-1,-1,1
pos_fov_ = np.concatenate([[[0]]*len(pos_fov),pos_fov[:,::transpose]*[flipx,flipy]],axis=-1)
cmsff = pos_fov_+cmsf

#pos_fov_[]
ifvsu,indexf = np.unique(ifovsf,return_index=True)

#dif_ = np.abs(np.diff(pos_fov_[indexf],axis=0))
#inter_distance_fov = np.min(dif_[dif_>50])

dd,ibest = KDTree(pos_fov_[indexf]+np.array([0,283.6,283.6])/2).query(cmsff)
keep = ifvsu[ibest]==ifovsf

cmsfff = cmsff[keep]
ifovsf = ifovsf[keep]
volmsf = volmsf[keep]
icells = icells[keep]

# save the final/filtered cell info
np.savez(r"\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\cellinfos_final.npz",cmsf=cmsfff,volmsf=volmsf,ifovsf=ifovsf,icells=icells)

#### *3b. Create annData object with marker gene expression*

In [ ]:
# load cell info
dic_info = np.load(r"\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\cellinfos_final.npz")
#,cmsf=cmsfff,volmsf=volmsf,ifovsf=ifovsf,icells=icells)
Xcells = dic_info['cmsf']
volms = dic_info['volmsf']
ifovs = dic_info['ifovsf']
icells = dic_info['icells']
ucells = np.array([f'Conv_zscan{ifov//10**4}_'+str(ifov%10**4).zfill(3)+'--'+str(icell) for ifov,icell in zip(ifovs,icells)])
dic_zxyvolm = {cell:(z,x,y,0,0,0,volm)for cell,(z,x,y),volm in zip(ucells,Xcells,volms)}

In [ ]:
# load final marker gene transcripts
fls = glob.glob(r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\final_spots_combined\*--MERcomb.npz')

M = None
cells = []

for fl in tqdm(fls):
    dic = np.load(fl)
    M_ = dic['Mmer']
    genes = dic['genes_mer']
    cells_ = dic['cells_mer']
    cells.extend(cells_)
    M = M_ if M is None else np.concatenate([M,M_])

cells = np.array(cells)

is_good = np.array([cell in dic_zxyvolm for cell in cells])
cellsf = cells[is_good]
Mf = M[is_good]
zxyvolm = np.array([dic_zxyvolm[cell] for cell in cellsf])

df = pd.DataFrame(data=Mf,
    index=cellsf,
    columns=genes)

cell_df = pd.DataFrame(data=zxyvolm,
    index=cellsf,
    columns=['zc','xc','yc','zfov','xfov','yfov','vol'])

In [ ]:
import pandas as pd, numpy as np, glob, os
import scanpy as sc

base_volm = 4*np.pi*10**3/3
dff=df
genes = [gn for gn in dff.columns if '_smFISH' not in gn and 'blank' not in gn]
dfR_ = dff[genes].copy()
dfR_ = dfR_.replace(np.nan, 0)
cell_dfR_ = cell_df

#keep = cell_dfR['volm']>th_vol
#dfR_ = dfR_.loc[keep]
#cell_dfR_ = cell_dfR.loc[keep]

scdata2 = sc.AnnData(dfR_) # create initial AnnData object
xcells = np.array(cell_df['xc'])
ycells = np.array(cell_df['yc'])
Xcells = np.array([xcells,ycells]).T
scdata2.obsm["X_spatial"] = Xcells
scdata2.obsm["X_raw"] = scdata2.X.copy()
scdata2.obs['volm']=cell_dfR_['vol']+base_volm

sc.pp.calculate_qc_metrics(scdata2, percent_top=None, inplace=True)
sc.pp.normalize_total(scdata2, target_sum=np.median(scdata2.obs["total_counts"]))
sc.pp.log1p(scdata2)

tot_counts = np.array(scdata2.obs["total_counts"])
tot_counts = tot_counts[(~np.isinf(tot_counts))&(~np.isnan(tot_counts))]

In [ ]:
scdata2 = scdata2[scdata2.obs["total_counts"]>75] # filter cells with low counts

# cluster the cells
sc.pp.neighbors(scdata2,use_rep='X')  #metric='correlation'
sc.tl.umap(scdata2,random_state=0,min_dist=0.1)
sc.tl.leiden(scdata2, resolution=3) #### 
cmap = ["#e6194B", "#3cb44b", "#ffe119", "#4363d8", "#f58231", "#911eb4", "#42d4f4", "#f032e6", "#bfef45",
        "#fabed4", "#469990", "#dcbeff", "#9A6324", "#fffac8", "#800000", "#aaffc3", "#808000", "#ffd8b1",
        "#000075", "#a9a9a9"]

scdata2.uns['cmap']=cmap

leiden = np.array(scdata2.obs['leiden'],dtype=int)

# write h5ad file
scdata2.write_h5ad('scdata_5_28_2025_BRBB500gn.h5ad')

#### *3c. Add cell type annotations to annData object*

In [ ]:
# load cell type annotations from cell type mapper output
abc_fl = r'scdata_5_28_2025_BRBB500gn_10xWholeMouseBrain(CCN20230722)_HierarchicalMapping_UTC_1752335763145.csv'
df_cell_type = pd.read_csv(abc_fl,skiprows=4)
scdata2.obs['cluster_name'] = np.array(df_cell_type['cluster_name']).astype(str)#subclass_name
scdata2.obs['cluster_bootstrapping_probability'] = np.array(df_cell_type['cluster_bootstrapping_probability']).astype(np.float32)
scdata2.obs['supertype_name'] = np.array(df_cell_type['supertype_name']).astype(str)
scdata2.obs['supertype_bootstrapping_probability'] = np.array(df_cell_type['supertype_bootstrapping_probability']).astype(np.float32)
scdata2.obs['subclass_name'] = np.array(df_cell_type['subclass_name']).astype(str)
scdata2.obs['subclass_bootstrapping_probability'] = np.array(df_cell_type['subclass_bootstrapping_probability']).astype(np.float32)
scdata2.obs['class_name'] = np.array(df_cell_type['class_name']).astype(str)
scdata2.obs['class_bootstrapping_probability'] = np.array(df_cell_type['class_bootstrapping_probability']).astype(np.float32)

In [ ]:
# write final h5ad file with cell type annotations
scdata2.write_h5ad('scdata_5_28_2025_BRBB500gn_final.h5ad')

#### *3d. Add cCRE transcripts to annData object*

In [ ]:
fls = glob.glob(r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\final_spots_combined\*--CREcomb2.npz')

M = None
cells = []

for fl in tqdm(fls):
    dic = np.load(fl)
    M_ = dic['Mmer']
    genes = dic['genes_mer']
    cells_ = dic['cells_mer']
    cells.extend(cells_)
    M = M_ if M is None else np.concatenate([M,M_])

cells = np.array(cells)
df = pd.DataFrame(data=M,
    index=cells,
    columns=genes)

In [ ]:
scdata = scdata2 # use the previous scdata with cell type annotations
cells_scdata = scdata.obs.index
cres = list(df.columns)
dic_cre = {cell:np.array(df.loc[cell]) for cell in df.index};
counts_all_cre = np.array([dic_cre.get(cell,np.zeros(len(cres))) for cell in cells_scdata])

In [ ]:
import pandas as pd

# write new h5ad file with cCRE counts added
scdata.obsm['CRE']=pd.DataFrame(counts_all_cre,columns=cres,index=cells_scdata)
scdata.write_h5ad('scdata_5_28_2025_BRBB500gn_final_CRE.h5ad',compression='gzip')

#### *3e. Add T7 transcripts to annData object*

In [ ]:
fls = glob.glob(r'\\192.168.0.115\starrfish3\Zane\Zane400Enhancers_WT_05_07_2025\final_spots_combined\*--TCREcomb3.npz')

M = None
cells = []
for fl in tqdm(fls):
    dic = np.load(fl)
    M_ = dic['Mmer']
    genes = dic['genes_mer']
    cells_ = dic['cells_mer']
    cells.extend(cells_)
    M = M_ if M is None else np.concatenate([M,M_])
cells = np.array(cells)

df = pd.DataFrame(data=M,
    index=cells,
    columns=genes)

In [ ]:
scdata = scdata2
cells_scdata = scdata.obs.index
cres = list(df.columns)
dic_cre = {cell:np.array(df.loc[cell]) for cell in df.index};
counts_all_cre = np.array([dic_cre.get(cell,np.zeros(len(cres))) for cell in cells_scdata])

In [ ]:
import pandas as pd

# write new h5ad file with T7CRE counts added
scdata.obsm['T7CRE']=pd.DataFrame(counts_all_cre,columns=cres,index=cells_scdata)
scdata.write_h5ad('scdata_5_28_2025_BRBB500gn_final_CRE_T7CRE.h5ad',compression='gzip')